<a href="https://colab.research.google.com/github/ga426553-sudo/Classification-of-Breast-Cancer-Subtypes-machine-learning---CuMiDa-22820/blob/main/No_7_XGBoost_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Séptimo código - Cáncer de Mama 🌸

**Implementación de XGBoost para Clasificación de Cáncer de Mama**

### Conectar con Google Drive 🌺

In [ ]:
# ============================================
# CÓDIGO 7: XGBOOST Y EVALUACIÓN
# ============================================

# -*- coding: utf-8 -*-
"""XGBoost con búsqueda de hiperparámetros y CV repetida"""

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pickle
import xgboost as xgb
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🔬 XGBOOST: BÚSQUEDA HIPERPARÁMETROS + CV REPETIDA (10x5)")
print("="*60)

Mounted at /content/drive
🔬 XGBOOST: BÚSQUEDA HIPERPARÁMETROS + CV REPETIDA (10x5)


### Cargar datos 🌺

In [ ]:
# 1. CARGAR DATOS
# ================
with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/subsets.pkl', 'rb') as f:
    subsets = pickle.load(f)
y_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_final.npy')

print(f"✅ Datos cargados. Subconjuntos: {list(subsets.keys())}")

✅ Datos cargados. Subconjuntos: ['EO', 'BBA', 'CSA', 'RDA', 'GA']


###Configurar XGBoost 🌺

In [ ]:
# 2. CONFIGURAR XGBOOST (según Tabla 4 del artículo)
# ====================
print("\n" + "="*60)
print("📌 BÚSQUEDA DE HIPERPARÁMETROS (GridSearchCV) en subconjunto EO")
print("="*60)

X_train_EO = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_EO.npy')
print(f"X_train_EO shape: {X_train_EO.shape}")

# Definir cuadrícula de parámetros
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.1, 0.2],
    'n_estimators': [100, 150],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
    'reg_lambda': [0.5, 1]
}

xgb_base = xgb.XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)

# GridSearchCV con 5-fold CV simple (suficiente para búsqueda)
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,  # 5-fold para búsqueda (rápido)
    verbose=1,
    n_jobs=-1
)

print("🚀 Iniciando búsqueda...")
grid_search.fit(X_train_EO, y_train)

print("\n🏆 Mejores parámetros encontrados:")
best_params = grid_search.best_params_
for k, v in best_params.items():
    print(f"   {k}: {v}")
print(f"Mejor accuracy CV (5-fold): {grid_search.best_score_:.4f}")



📌 BÚSQUEDA DE HIPERPARÁMETROS (GridSearchCV) en subconjunto EO
X_train_EO shape: (206, 3)
🚀 Iniciando búsqueda...
Fitting 5 folds for each of 96 candidates, totalling 480 fits

🏆 Mejores parámetros encontrados:
   colsample_bytree: 0.8
   learning_rate: 0.1
   max_depth: 3
   n_estimators: 100
   reg_lambda: 1
   subsample: 0.8
Mejor accuracy CV (5-fold): 0.9469


In [ ]:
# 3. CONFIGURAR MODELO FINAL CON LOS MEJORES PARÁMETROS
xgb_final = xgb.XGBClassifier(**best_params, random_state=42, eval_metric='logloss', use_label_encoder=False, objective='binary:logistic')

###Mostrar resultados 🌺

In [ ]:
# 4. EVALUACIÓN CON CV REPETIDA (10 folds, 5 repeticiones) PARA CADA SUBSET
print("\n" + "="*60)
print("📊 EVALUACIÓN CON CV REPETIDA (10x5) PARA CADA SUBSET")
print("="*60)

rkf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=42)
results = {}

for name in subsets.keys():
    X_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_{name}.npy')
    acc = cross_val_score(xgb_final, X_subset, y_train, cv=rkf, scoring='accuracy')
    f1 = cross_val_score(xgb_final, X_subset, y_train, cv=rkf, scoring='f1')
    auc = cross_val_score(xgb_final, X_subset, y_train, cv=rkf, scoring='roc_auc')
    results[name] = {
        'accuracy': f"{acc.mean():.4f} ± {acc.std():.4f}",
        'f1': f"{f1.mean():.4f} ± {f1.std():.4f}",
        'auc': f"{auc.mean():.4f} ± {auc.std():.4f}"
    }
    print(f"\n{name}:")
    print(f"   Accuracy: {results[name]['accuracy']}")
    print(f"   F1: {results[name]['f1']}")
    print(f"   AUC: {results[name]['auc']}")



📊 EVALUACIÓN CON CV REPETIDA (10x5) PARA CADA SUBSET

EO:
   Accuracy: 0.9397 ± 0.0488
   F1: 0.9365 ± 0.0517
   AUC: 0.9875 ± 0.0149

BBA:
   Accuracy: 0.8845 ± 0.0601
   F1: 0.8802 ± 0.0644
   AUC: 0.9561 ± 0.0330

CSA:
   Accuracy: 0.9138 ± 0.0551
   F1: 0.9126 ± 0.0558
   AUC: 0.9620 ± 0.0476

RDA:
   Accuracy: 0.8085 ± 0.0728
   F1: 0.7886 ± 0.0840
   AUC: 0.8539 ± 0.0769

GA:
   Accuracy: 0.6936 ± 0.1011
   F1: 0.6743 ± 0.1162
   AUC: 0.7928 ± 0.0981


###Indentificar mejor modelo 🌺

In [ ]:
# 5. GUARDAR RESULTADOS
# ================================================
with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/results_xgboost_cv_repeat.pkl', 'wb') as f:
    pickle.dump(results, f)

print("\n✅ Resultados guardados en 'results_xgboost_cv_repeat.pkl'")


✅ Resultados guardados en 'results_xgboost_cv_repeat.pkl'


### Identificar el Mejor Modelo 🌺

In [ ]:
# 6. IDENTIFICAR EL MEJOR SUBCONJUNTO BASADO EN ACCURACY
# ====================================================

best_accuracy = -1
best_subset_name = None
best_subset_results = None

for name, res in results.items():
    # Extraer el valor medio de accuracy (eliminar el ± std)
    current_accuracy = float(res['accuracy'].split(' ')[0])
    if current_accuracy > best_accuracy:
        best_accuracy = current_accuracy
        best_subset_name = name
        best_subset_results = res

print(f"\n🏆 Mejor subconjunto identificado (basado en Accuracy media de CV repetida): {best_subset_name}")
print(f"   Accuracy: {best_subset_results['accuracy']}")
print(f"   F1: {best_subset_results['f1']}")
print(f"   AUC: {best_subset_results['auc']}")


🏆 Mejor subconjunto identificado (basado en Accuracy media de CV repetida): EO
   Accuracy: 0.9397 ± 0.0488
   F1: 0.9365 ± 0.0517
   AUC: 0.9875 ± 0.0149
